# 1. 短期记忆

In [ ]:
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
import os
from langgraph.checkpoint.memory import InMemorySaver
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv(".env")

model = init_chat_model(
    model="deepseek-v4-pro",
    model_provider="deepseek",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL"),
)

checkpointer = InMemorySaver()
config = {"configurable": {"thread_id": "1"}}

agent = create_agent(model=model, checkpointer=checkpointer)

resp = agent.invoke(
    {"messages": [HumanMessage("你好，我叫aihaipeng，喜欢仓鼠")]}, config=config
)

for msg in resp["messages"]:
    msg.pretty_print()

In [ ]:
resp = agent.invoke(
    {"messages": [HumanMessage("还记的我的名字和喜欢的小动物么？")]}, config=config
)
for msg in resp["messages"]:
    msg.pretty_print()

# 2. 基于外部存储介质的持久化

In [ ]:
from langgraph.checkpoint.postgres import PostgresSaver
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
import os
from langgraph.checkpoint.memory import InMemorySaver
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv(".env")

model = init_chat_model(
    model="deepseek-v4-pro",
    model_provider="deepseek",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL"),
)

DB_URL = os.environ["DATABASE_URL"]

with PostgresSaver.from_conn_string(DB_URL) as checkpointer:
    print("已连接 PostgreSQL")
    checkpointer.setup()
    print("checkpoint 表已就绪")

    agent = create_agent(model=model, checkpointer=checkpointer)
    config = {"configurable": {"thread_id": "1"}}

    print("开始调用模型")
    resp = agent.invoke(
        {"messages": [HumanMessage("你好，我是艾海鹏，喜欢仓鼠")]},
        config=config,
    )

    for msg in resp["messages"]:
        msg.pretty_print()

    resp = agent.invoke(
        {"messages": [HumanMessage("还记的我的名字和喜欢的小动物么？")]},
        config=config,
    )

    for msg in resp["messages"]:
        msg.pretty_print()

# 3. 记忆的治理策略（上下文管理）

## 3.1 消息裁剪
- 调用模型前裁剪上下文，通常保留系统初始消息和最近若干消息，或按token数保留末尾内容  
- 适合成本敏感，对旧上下文依赖不强的场景

### 1. 视图裁剪
只裁"发给模型的视图",不动 state(推荐) → 用 wrap_model_call + request.override(messages=...)。原始历史仍完整保存在 checkpointer 里,只是每次调用时给模型一份裁过的副本。安全、可逆。

**核心参数：**

| 参数 | 作用 | 建议默认值 |
| --- | --- | --- |
| **messages** | 待裁剪的消息列表 | request.messages |
| **max_tokens** | token 预算上限，超过才触发裁剪 | 按模型上下文窗口的 60%~80% 设置 |
| **strategy** | "last" 保留末尾 / "first" 保留开头 | "last"（对话场景几乎都用这个） |
| **token_counter** | 计数方式：模型实例（准但慢）/ count_tokens_approximately（快）/ len（按条数） | count_tokens_approximately |
| **include_system** | 是否始终保留 SystemMessage，不参与裁剪 | True |
| **start_on** | 裁剪后第一条消息必须是指定类型，避免以 ToolMessage 或 AIMessage 开头破坏依赖链 | "human" |
| **end_on** | 裁剪后最后一条消息必须是指定类型，避免以含 tool_calls 的 AIMessage 结尾留下未完成的工具调用 | ("human", "tool") |
| **allow_partial** | 是否允许截断单条消息的部分内容（如只保留后半段文本） | False |

**start_on 和 end_on 的选值逻辑：**

**核心矛盾：消息之间有"依赖关系"**

AIMessage 里的 **tool_calls** 和后面的 **ToolMessage** 是一对绑定关系：


AIMessage(tool_calls=[{name:"search", id:"call_1"}])  ← 发起调用  
ToolMessage(content="结果", tool_call_id="call_1")     ← 必须紧跟在后面


trim_messages(strategy="last") 保留的是**连续的尾部窗口**，不会在中间挖洞。所以斩断只可能发生在窗口的**头尾两个边界**：头部可能留下孤立的 ToolMessage（发起它的 AIMessage 被切在窗口外），尾部可能留下孤立的 AIMessage(tool_calls)（它的 ToolMessage 还在窗口外）。**start_on** 管头，**end_on** 管尾。

**start_on="human" —— 为什么开头建议是 HumanMessage**

裁剪后第一条消息如果是：

| 开头消息类型 | 后果 |
| --- | --- |
| **ToolMessage**（孤立） | 发起它的 AIMessage(tool_calls) 被切在窗口外，后向约束违反（tool 找不到对应的 assistant 调用），❌ **必报错** |
| **AIMessage(tool_calls=...)** | 配对的 ToolMessage 因连续窗口通常仍在其后，前向约束满足，**多数能跑通**；但跨 provider 容忍度不一，start_on="human" 保守起见仍排除它 |
| **HumanMessage** | 自包含，不依赖任何前序消息，✅ 绝对安全 |

设 start_on="human" 真正要防的是**开头出现孤立的 ToolMessage**；它顺带也排除了本来多半合法的 AIMessage(tool_calls) 开头，属于"一刀切保守策略"，换来语义整洁和跨 provider 稳定。

**end_on=("human", "tool") —— 为什么结尾不能是 AI**

裁剪后最后一条消息如果是：

| 结尾消息类型 | 后果 |
| --- | --- |
| **AIMessage(tool_calls=...)** | 模型说了要调工具，但这是最后一条，工具还没调，序列不完整，❌ 报错 |
| **AIMessage(content="正常回复")** | 没有 tool_calls 的纯文本回复，可以，但 trim_messages 不区分有无 tool_calls，保守起见排除 |
| **HumanMessage** | 用户提问，模型下一步就是回答，✅ 安全 |
| **ToolMessage** | 工具结果，模型下一步基于结果推理，✅ 安全 |

设 end_on=("human", "tool") = 最后一条留个"等模型接话"的状态，不会留下未完成的 tool_call。

In [ ]:
from typing import Callable

from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
from langchain.agents.middleware import AgentMiddleware, ModelRequest, ModelResponse
from langchain_core.messages import trim_messages
from langchain_core.messages.utils import count_tokens_approximately
import os
from langgraph.checkpoint.memory import InMemorySaver
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from rich import print as rprint

load_dotenv(".env")

model = init_chat_model(
    model="deepseek-v4-pro",
    model_provider="deepseek",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL"),
)

# 按Toekn裁剪
class ContextManagerMiddleware(AgentMiddleware):
    def __init__(self, max_tokens=4000, strategy="last"):
        super().__init__()
        self.max_tokens = max_tokens
        self.strategy = strategy

    def wrap_model_call(
        self, request: ModelRequest, handler: Callable[[ModelRequest], ModelResponse]
    ) -> ModelResponse:
        trimmed = trim_messages(
            request.messages,
            max_tokens=self.max_tokens,
            strategy=self.strategy,
            token_counter=count_tokens_approximately,
            include_system=True,
            start_on="human",
            end_on=("human", "tool"),
            allow_partial=False,
        )
        return handler(request.override(messages=trimmed))


checkpointer = InMemorySaver()

config = {"configurable": {"thread_id": "1"}}

agent = create_agent(
    model=model,
    middleware=[ContextManagerMiddleware(max_tokens=100)],
    checkpointer=checkpointer,
)

agent.invoke({"messages": [HumanMessage("你好，我是aihaipeng")]}, config)
agent.invoke({"messages": [HumanMessage("从现在起，你叫小王")]}, config)
agent.invoke({"messages": [HumanMessage("今天天气不错")]}, config)

resp = agent.invoke({"messages": [HumanMessage("告诉我你是谁？我是谁？")]}, config)

rprint(resp)

In [ ]:
from typing import Callable

from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
from langchain.agents.middleware import AgentMiddleware, ModelRequest, ModelResponse
from langchain_core.messages import trim_messages
from langchain_core.messages.utils import count_tokens_approximately
import os
from langgraph.checkpoint.memory import InMemorySaver
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from rich import print as rprint

load_dotenv(".env")

model = init_chat_model(
    model="deepseek-v4-pro",
    model_provider="deepseek",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL"),
)

# 按消息条数裁剪
class MessageCountManagerMiddleware(AgentMiddleware):
    def __init__(self, max_messages=6, strategy="last"):
        super().__init__()
        self.max_messages = max_messages   # 语义从"token 预算"变成"保留条数"
        self.strategy = strategy

    def wrap_model_call(
        self, request: ModelRequest, handler: Callable[[ModelRequest], ModelResponse]
    ) -> ModelResponse:
        trimmed = trim_messages(
            request.messages,
            max_tokens=self.max_messages,   # token_counter=len 时，这个数就是"保留几条"
            strategy=self.strategy,
            token_counter=len,              # ← 唯一的核心改动：按消息条数计数
            include_system=True,
            start_on="human",
            end_on=("human", "tool"),
            allow_partial=False,            # 按条数裁只能整条留/整条删，保持 False
        )
        return handler(request.override(messages=trimmed))



checkpointer = InMemorySaver()

config = {"configurable": {"thread_id": "2"}}

agent = create_agent(
    model=model,
    middleware=[MessageCountManagerMiddleware(max_messages=4)],
    checkpointer=checkpointer,
)

agent.invoke({"messages": [HumanMessage("你好，我是aihaipeng")]}, config)
agent.invoke({"messages": [HumanMessage("从现在起，你叫小王")]}, config)
agent.invoke({"messages": [HumanMessage("今天天气不错")]}, config)

resp = agent.invoke({"messages": [HumanMessage("告诉我你是谁？我是谁？")]}, config)

rprint(resp)

### 2. state 裁剪

state 裁剪：真正从 state 删除历史消息，不可逆。用 before_model + RemoveMessage 实现。

**机制**

**before_model** 是 Node-style hook，返回的 dict 会更新 state。要"真删"，靠一个特殊哨兵 REMOVE_ALL_MESSAGES：

- RemoveMessage(id=REMOVE_ALL_MESSAGES) 先把 state 里的消息全部清空
- 后面跟上 *trimmed 再把裁剪后的消息写回去

合起来就是"清空 + 重写"，把 state 的消息列表替换成裁剪后的短列表，checkpointer 存的也随之变短。

**和视图裁剪的关键区别**

| 维度 | 视图裁剪（### 1） | state 裁剪（本节） |
| --- | --- | --- |
| 用哪个 hook | **wrap_model_call** | **before_model** |
| 读的数据 | request.messages | state["messages"] |
| 后半步 | request.override(messages=) | 返回 RemoveMessage |
| state 变不变 | 不变，完整保留 | 真的变短 |
| resp["messages"] | 完整 | 变短 |
| 可逆性 | 可逆 | 不可逆 |
| 下一轮基于什么裁 | 每轮从完整历史重裁 | 在已删的基础上继续 |

**累积效应**：这一轮删掉的，下一轮 state["messages"] 里就真没了，trim_messages 的输入是已经被削短的历史。视图裁剪则是每轮都从完整历史重裁——这正是"可逆 vs 不可逆"的根源。

In [ ]:
from typing import Any

from langchain.agents import AgentState, create_agent
from langchain.agents.middleware import AgentMiddleware
from langchain_core.messages import HumanMessage, RemoveMessage, trim_messages
from langchain_core.messages.utils import count_tokens_approximately
from langgraph.graph.message import REMOVE_ALL_MESSAGES  # 清空 state 的哨兵
from langgraph.runtime import Runtime
import os
from langgraph.checkpoint.memory import InMemorySaver
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from rich import print as rprint

load_dotenv(".env")

model = init_chat_model(
    model="deepseek-v4-pro",
    model_provider="deepseek",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL"),
)


# state 裁剪：真正删除 state 里的历史消息
class StateTrimMiddleware(AgentMiddleware):
    """按token裁剪 state"""

    def __init__(self, max_tokens=4000, strategy="last"):
        super().__init__()
        self.max_tokens = max_tokens
        self.strategy = strategy

    def before_model(
        self, state: AgentState, runtime: Runtime
    ) -> dict[str, Any] | None:
        trimmed = trim_messages(
            state["messages"],  # 注意：读的是 state，不是 request
            max_tokens=self.max_tokens,
            strategy=self.strategy,
            token_counter=count_tokens_approximately,
            include_system=True,
            start_on="human",
            end_on=("human", "tool"),
            allow_partial=False,
        )
        # 没超预算、没裁掉东西就不动 state，避免每轮做无意义的清空重写
        if len(trimmed) == len(state["messages"]):
            return None
        # 清空全部 + 写回裁剪后的 → 真正替换 state
        return {"messages": [RemoveMessage(id=REMOVE_ALL_MESSAGES), *trimmed]}


checkpointer = InMemorySaver()

config = {"configurable": {"thread_id": "3"}}

agent = create_agent(
    model=model,
    middleware=[StateTrimMiddleware(max_tokens=20)],
    checkpointer=checkpointer,
)

agent.invoke({"messages": [HumanMessage("你好，我是aihaipeng")]}, config)
agent.invoke({"messages": [HumanMessage("从现在起，你叫小王")]}, config)
agent.invoke({"messages": [HumanMessage("今天天气不错")]}, config)

resp = agent.invoke({"messages": [HumanMessage("告诉我你是谁？我是谁？")]}, config)

# 对照：视图裁剪时 resp["messages"] 完整；state 裁剪时这里会明显变短
rprint(resp)

In [ ]:
from typing import Any

from langchain.agents import AgentState, create_agent
from langchain.agents.middleware import AgentMiddleware
from langchain_core.messages import HumanMessage, RemoveMessage, trim_messages
from langgraph.graph.message import REMOVE_ALL_MESSAGES
from langgraph.runtime import Runtime
from langgraph.checkpoint.memory import InMemorySaver
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from rich import print as rprint
import os

load_dotenv(".env")

model = init_chat_model(
    model="deepseek-v4-pro",
    model_provider="deepseek",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL"),
)


class MessageCountTrimMiddleware(AgentMiddleware):
    """按消息条数裁剪 state"""

    def __init__(self, max_messages=4, strategy="last"):
        super().__init__()
        self.max_messages = max_messages
        self.strategy = strategy

    def before_model(
        self, state: AgentState, runtime: Runtime
    ) -> dict[str, Any] | None:

        messages = state["messages"]

        trimmed = trim_messages(
            messages,
            max_tokens=self.max_messages,
            strategy=self.strategy,
            token_counter=len,  # token_counter 改成 len，这样 max_tokens 实际表示“最多保留几条消息”
            include_system=True,
            start_on="human",
            end_on=("human", "tool"),
            allow_partial=False,
        )

        print(f"原消息数：{len(messages)}，裁剪后：{len(trimmed)}")

        # 没有发生裁剪
        if len(trimmed) == len(messages):
            return None

        # 清空旧 state，再写入裁剪后的消息
        return {
            "messages": [
                RemoveMessage(id=REMOVE_ALL_MESSAGES),
                *trimmed,
            ]
        }


checkpointer = InMemorySaver()

config = {"configurable": {"thread_id": "message-count-demo"}}

agent = create_agent(
    model=model,
    middleware=[MessageCountTrimMiddleware(max_messages=4)],
    checkpointer=checkpointer,
)


agent.invoke(
    {"messages": [HumanMessage("你好，我是 aihaipeng")]},
    config,
)
agent.invoke(
    {"messages": [HumanMessage("从现在起，你叫小王")]},
    config,
)
agent.invoke(
    {"messages": [HumanMessage("今天天气不错")]},
    config,
)

resp = agent.invoke(
    {"messages": [HumanMessage("告诉我你是谁？我是谁？")]},
    config,
)

rprint(resp)

## 3.2 消息删除

消息裁剪（3.1）和消息删除都会让历史变短，但发生的时机和出发点都不同。

裁剪是**预算驱动**、发生在**模型调用前**：在 **before_model** 或 **wrap_model_call** 里，按 token 数或条数保留一段连续的尾部窗口，收窄这次交给模型的上下文——它不关心某条消息说了什么，只决定"模型这次能看到多少"。

删除是**意图驱动**、通常发生在**模型调用后**：模型答完后，你发现某几条消息不该留（一次失败的工具调用、一段跑题的对话、一条含敏感信息的消息），在 **after_model** 里用 **RemoveMessage** 按消息的 id 精准点名删掉，把它们从 state 里永久移除——针对的是特定内容，而不是压缩长度。

**两者的区别**

| 维度 | 消息裁剪（3.1） | 消息删除（3.2） |
| --- | --- | --- |
| 发生时机 | 模型调用前（**before_model** / **wrap_model_call**） | 模型调用后（**after_model**）为主 |
| 触发依据 | 上下文预算：token 数 / 条数超限 | 业务意图：这条消息该不该留 |
| 选择粒度 | 按位置连续批量取舍（保留一个窗口） | 按 id 精准点名单条 |
| 是否看内容 | 不看，只按位置和预算 | 看，针对特定内容 |
| 目的 | 控成本、防止超出上下文窗口 | 纠错、脱敏、去噪、撤回 |
| 视角 | 决定"模型这次能看到多少" | 决定"哪几条不该留在历史里" |

**使用场景**

- 消息裁剪：长期运行的对话 agent，历史不断增长、成本敏感，且旧上下文依赖不强——保住最近的窗口就够用。
- 消息删除：
  - 模型答完后，删掉一次报错或跑偏的工具调用，连同它的 **ToolMessage**，避免污染后续推理
  - 用户要求撤回、删除刚才说过的某条消息
  - 脱敏，移除包含密码、密钥、隐私的消息
  - 清理一段无效或跑题的对话

**一点澄清**：3.1 的 state 裁剪其实也会永久改 state，所以"改不改状态"不是两者的分界；真正的分界是**时机 + 目的**——裁剪在调用前、为收窄本次输入服务（按窗口整段重写），删除在调用后、为清掉不该留的内容服务（按 id 精确移除）。两者落到 state 时都用 **RemoveMessage**：裁剪配合 **REMOVE_ALL_MESSAGES** 哨兵清空重写整个窗口，删除只传目标消息真实的 id。

下面用一个脱敏中间件演示"按 id 精准删除"：在 **after_model** 里逐条检查历史，命中敏感词的消息，用 **RemoveMessage** 按它自己的 id 删掉。和裁剪最大的不同是——它看内容、只删命中的那几条，不关心窗口大小。

一个要点：删带 **tool_calls** 的 AIMessage 时，必须连同它配对的 **ToolMessage** 一起删，否则会留下断裂的依赖链（参见 3.1 的选值逻辑）。纯文本消息删单条通常是安全的。

In [1]:
from typing import Any

from langchain.agents import AgentState, create_agent
from langchain.agents.middleware import AgentMiddleware
from langchain_core.messages import HumanMessage, RemoveMessage
from langgraph.runtime import Runtime
import os
from langgraph.checkpoint.memory import InMemorySaver
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from rich import print as rprint

load_dotenv(".env")

model = init_chat_model(
    model="deepseek-v4-pro",
    model_provider="deepseek",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL"),
)

# 命中这些词就算敏感信息
SENSITIVE = ["密码", "身份证", "银行卡"]


def is_sensitive(content) -> bool:
    return isinstance(content, str) and any(w in content for w in SENSITIVE)


# 脱敏中间件：模型回复后，把历史里命中敏感词的消息按 id 删掉
class RedactMiddleware(AgentMiddleware):
    def after_model(
        self, state: AgentState, runtime: Runtime
    ) -> dict[str, Any] | None:
        # 看内容：逐条检查，挑出命中敏感词的消息 id
        hit_ids = [msg.id for msg in state["messages"] if is_sensitive(msg.content)]
        if not hit_ids:
            return None
        # 按 id 精准点名删除，只删命中的这几条，其余历史原样保留
        return {"messages": [RemoveMessage(id=mid) for mid in hit_ids]}


checkpointer = InMemorySaver()

config = {"configurable": {"thread_id": "redact-demo"}}

agent = create_agent(
    model=model,
    middleware=[RedactMiddleware()],
    checkpointer=checkpointer,
)

agent.invoke({"messages": [HumanMessage("你好，我叫 aihaipeng")]}, config)
agent.invoke({"messages": [HumanMessage("顺便记一下，我的密码是 abc123")]}, config)
resp = agent.invoke({"messages": [HumanMessage("我叫什么名字？")]}, config)

# 含"密码"的那条消息已被按 id 删除，这里打印的历史里看不到它
rprint(resp)

{
    'messages': [
        HumanMessage(
            content='你好，我叫 aihaipeng',
            additional_kwargs={},
            response_metadata={},
            id='96e85aee-5cf4-4125-8e8a-7f6c7b654933'
        ),
        AIMessage(
            content='你好，aihaipeng！很高兴认识你！😊\n\n我已经记住你的名字了。有什么我可以帮你的吗？无论是工作、学习，还是日常琐事，我都乐意效劳～',
            additional_kwargs={
                'refusal': None,
                'reasoning_content': '嗯，用户发来一句简单的自我介绍：“你好，我叫 aihaipeng”。\n\n这是一个很基础的社交开场，用户主动告知了自己的名字。我需要给出一个友好、热情的回应，确认我记住了这个名字，并表达出我随时准备提供帮助的态度。\n\n我可以先直接称呼对方的名字来回应问候，表示很高兴认识并记住了这个名字。然后表达我在这提供支持的意愿，最后用一个开放性的问题引导对话继续，比如询问今天有什么可以帮忙的。\n\n想到了用轻松愉快的语气，结尾加个表情符号显得更亲切自然。'
            },
            response_metadata={
                'token_usage': {
                    'completion_tokens': 152,
                    'prompt_tokens': 11,
                    'total_tokens': 163,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': 110,
                        'rejected_prediction_tokens': None
                    },
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0},
                    'prompt_cache_hit_tokens': 0,
                    'prompt_cache_miss_tokens': 11
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-v4-pro',
                'system_fingerprint': 'fp_9954b31ca7_prod0820_fp8_kvcache_20260402',
                'id': '2a8849c1-585f-4b6f-bb21-8d5486d1b076',
                'finish_reason': 'stop',
                'logprobs': None
            },
            id='lc_run--019f66c2-1117-75a3-a383-ea5a20d95704-0',
            tool_calls=[],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 11,
                'output_tokens': 152,
                'total_tokens': 163,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {'reasoning': 110}
            }
        ),
        HumanMessage(
            content='我叫什么名字？',
            additional_kwargs={},
            response_metadata={},
            id='8f5eac99-f858-448c-a921-86aac40ab28e'
        ),
        AIMessage(
            content='你叫 **aihaipeng**！  \n我之前已经记住啦，不会忘的～ 有什么新问题想聊吗？ 😄',
            additional_kwargs={
                'refusal': None,
                'reasoning_content': '我们需要确认用户问“我叫什么名字？”，而之前用户自我介绍说“我叫 aihaipeng”。我需要回答正确。\n\n根据对话历史，用户明确说“你好，我叫 aihaipeng”，我回复了并说“我已经记住你的名字了”。现在用户问“我叫什么名字？”，我应该直接回答：“你叫 aihaipeng。”\n\n注意，回答时要友好自然，确认之前记住的名字。不需要额外猜测或解释。'
            },
            response_metadata={
                'token_usage': {
                    'completion_tokens': 122,
                    'prompt_tokens': 60,
                    'total_tokens': 182,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': 92,
                        'rejected_prediction_tokens': None
                    },
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0},
                    'prompt_cache_hit_tokens': 0,
                    'prompt_cache_miss_tokens': 60
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-v4-pro',
                'system_fingerprint': 'fp_9954b31ca7_prod0820_fp8_kvcache_20260402',
                'id': '6a795213-cdc4-47dd-8357-eb4726232154',
                'finish_reason': 'stop',
                'logprobs': None
            },
            id='lc_run--019f66c2-4e4f-7d72-b8b7-8562c65dd3ec-0',
            tool_calls=[],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 60,
                'output_tokens': 122,
                'total_tokens': 182,
                'input_token_det